# Modelo: Momento Óptimo (Share of Wallet + Velocidad)
Este notebook implementa el cálculo del `Share of Wallet` y su `Velocidad` (derivada mensual), detectando alertas cuando la velocidad cae más de 5% mensual. Incluye generación de datos de ejemplo si no hay fichero de ventas disponible.

## 1) Importar librerías

In [4]:
# Imports con manejo de errores para entornos donde falten paquetes
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import os
    from datetime import datetime
    try:
        get_ipython().run_line_magic('matplotlib', 'inline')
    except Exception:
        # No estamos en un kernel interactivo, continuar sin magia
        pass
except ModuleNotFoundError as e:
    print('Falta un paquete requerido:', e)
    print('Instala dependencias: pip install pandas numpy matplotlib')
    raise

## 2) Cargar datos (o generar ejemplo)
Se espera un `DataFrame` con columnas: `customer_id`, `date` (datetime), `sales_eur`, `potencial_annual` (o `Potencial_EUR_anual`). Si no hay fichero, se genera un dataset sintético de ejemplo.

In [ ]:
# Ajusta esta ruta si tienes un CSV real con ventas mensuales
candidate_paths = [
    'data/sales.csv',
    'data/master_commodities.csv',
    'data/raw/Datasets.xlsx - Ventas.csv',
]
df = None
for p in candidate_paths:
    if os.path.exists(p):
        try:
            df = pd.read_csv(p, parse_dates=['date'], dayfirst=True)
            print(f'Loaded {p}')
            break
        except Exception as e:
            pass

if df is None:
    # Generar datos de ejemplo
    np.random.seed(42)
    customers = ['CUST_A', 'CUST_B', 'CUST_C']
    periods = pd.date_range(end=pd.Timestamp.today(), periods=36, freq='MS')
    rows = []
    for cust in customers:
        potencial = np.random.uniform(50000, 200000)  # potencial anual en EUR
        base = np.random.uniform(2000, 15000)
        trend = np.random.uniform(-200, 400)
        for i, d in enumerate(periods):
            seasonal = 1 + 0.1 * np.sin(2 * np.pi * (i % 12) / 12)
            noise = np.random.normal(0, base * 0.1)
            sales = max(0, base + trend * (i/12) + noise) * seasonal
            rows.append({'customer_id': cust, 'date': d, 'sales_eur': sales, 'potencial_annual': potencial})
    df = pd.DataFrame(rows)
    print('Generado dataset sintético con', df['customer_id'].nunique(), 'clientes y', df['date'].nunique(), 'meses')

ValueError: Invalid frequency: M. Failed to parse with error message: ValueError("'M' is no longer supported for offsets. Please use 'ME' instead.")

## 3) Preprocesado mensual y cálculo de rolling 12 meses

In [ ]:
# Asegurarnos de las columnas y tipos
df = df.copy()
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
else:
    raise ValueError('El dataset necesita una columna `date`.')
if 'potencial_annual' not in df.columns and 'Potencial_EUR_anual' in df.columns:
    df['potencial_annual'] = df['Potencial_EUR_anual']

# Agrupar ventas por cliente y mes (inicio de mes)
df_monthly = df.groupby(['customer_id', pd.Grouper(key='date', freq='MS')])['sales_eur'].sum().reset_index()
# Añadir potencial por cliente: tomar el máximo (debe ser anual)
potencial = df.groupby('customer_id')['potencial_annual'].max().rename('potencial_annual')
df_monthly = df_monthly.merge(potencial.reset_index(), on='customer_id', how='left')
# Ordenar y calcular rolling 12 meses
df_monthly = df_monthly.sort_values(['customer_id', 'date'])
df_monthly['rolling_12m'] = df_monthly.groupby('customer_id')['sales_eur'].rolling(window=12, min_periods=1).sum().reset_index(0, drop=True)
# Share of Wallet (fracción del potencial anual que nos dedican)
df_monthly['share'] = df_monthly['rolling_12m'] / df_monthly['potencial_annual']
df_monthly.head()

## 4) Calcular Velocidad del Share y detectar alertas

In [ ]:
# Velocidad = diferencia mes a mes (share_t - share_{t-1})
df_monthly['velocity_share'] = df_monthly.groupby('customer_id')['share'].diff().fillna(0)
# Umbral de caída: -0.05 (caída absoluta de 5 puntos en share, i.e., -5% del total)
threshold = -0.05
df_monthly['alert_fall'] = df_monthly['velocity_share'] < threshold
# Señalar último mes procesado por cliente
last_dates = df_monthly.groupby('customer_id')['date'].max().rename('last_date')
summary_last = df_monthly.merge(last_dates.reset_index(), on=['customer_id', 'date'], how='inner')
summary_last[['customer_id', 'date', 'rolling_12m', 'potencial_annual', 'share', 'velocity_share', 'alert_fall']]

## 5) Funciones de visualización y ejemplo por cliente

In [ ]:
def plot_customer_momentum(dfm, customer_id, figsize=(12,5), threshold=-0.05):
    d = dfm[dfm['customer_id'] == customer_id].set_index('date')
    fig, ax1 = plt.subplots(figsize=figsize)
    ax1.plot(d.index, d['share'], marker='o', color='tab:blue', label='Share (rolling 12m / potencial)')
    ax1.set_ylabel('Share (fraction of potencial)')
    ax1.set_ylim(0, max(1.0, d['share'].max()*1.2))

    ax2 = ax1.twinx()
    colors = ['green' if v >= 0 else 'red' for v in d['velocity_share']]
    ax2.bar(d.index, d['velocity_share'], color=colors, alpha=0.6, width=20, label='Velocidad del Share')
    ax2.set_ylabel('Velocidad (delta mensual de share)')
    ax2.axhline(threshold, color='red', linestyle='--', linewidth=1, label=f'Umbral alerta ({threshold})')

    # Marcar alertas
    alerts = d[d['velocity_share'] < threshold]
    if not alerts.empty:
        ax1.scatter(alerts.index, alerts['share'], color='red', s=80, edgecolor='k', label='Alerta caída > umbral')

    ax1.set_title(f'Cliente: {customer_id} — Share y Velocidad')
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')
    plt.show()

# Ejemplo: plot para el primer cliente disponible
example_customer = df_monthly['customer_id'].unique()[0]
plot_customer_momentum(df_monthly, example_customer, threshold=threshold)

## 6) Resumen de alertas para la acción comercial

In [ ]:
# Lista de clientes con alerta en su último mes
alerts = summary_last[summary_last['alert_fall']].copy()
if alerts.empty:
    print('No hay alertas de caída > 5% en el último mes para ningún cliente (con los datos actuales).')
else:
    display_cols = ['customer_id', 'date', 'rolling_12m', 'potencial_annual', 'share', 'velocity_share']
    print('Clientes con alerta (último mes):')
    display(alerts[display_cols].sort_values('velocity_share'))

## 7) Interpretación y acciones recomendadas
- **Velocidad positiva (barras verdes):** Cliente en ciclo de crecimiento — acción: fidelizar y cross-sell.
- **Velocidad negativa leve:** Monitorizar y preparar contacto preventivo.
- **Velocidad negativa fuerte (caída > 5% mensual):** Alerta inmediata — contactar vía comercial, oferta agresiva o intervención personalizada.

Puedes ajustar el umbral `threshold` más arriba o debajo según tolerancia al riesgo, y automatizar notificaciones (email/CRM) cuando `alert_fall` sea True en el último mes.